# 🧠 Important Neural Network Types
## A Complete Guide — Architecture, Working, and Usage

> Each section covers: **What it is → How it works → Key components → Use cases → Code example**

---
| # | Network Type | Short Name | Best For |
|---|---|---|---|
| 1 | Perceptron / Multi-Layer Perceptron | MLP | Tabular data, classification |
| 2 | Convolutional Neural Network | CNN | Images, spatial data |
| 3 | Recurrent Neural Network | RNN | Sequences, time-series |
| 4 | Long Short-Term Memory | LSTM | Long sequences, NLP |
| 5 | Transformer | — | LLMs, translation, vision |
| 6 | Autoencoder | AE | Compression, anomaly detection |
| 7 | Generative Adversarial Network | GAN | Image synthesis, data augmentation |
| 8 | Graph Neural Network | GNN | Graphs, molecules, social networks |
| 9 | Diffusion Model | — | Image/audio generation |
|10 | Reinforcement Learning Networks | DQN/PPO | Games, robotics, agents |


## ⚙️ Setup — Install & Import Dependencies

In [ ]:
# Install dependencies (run once)
# !pip install numpy matplotlib torch torchvision scikit-learn

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams['figure.facecolor'] = '#0f0f0f'
plt.rcParams['axes.facecolor']   = '#1a1a2e'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = 'white'
plt.rcParams['xtick.color']      = 'white'
plt.rcParams['ytick.color']      = 'white'
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False

print("✅ Imports ready!")


---
## 1. 🔵 Perceptron & Multi-Layer Perceptron (MLP)

### What is it?
The **Perceptron** is the simplest neural network unit — a single neuron.  
The **MLP** stacks multiple layers of neurons: *Input → Hidden Layer(s) → Output*.

### How it works
1. Each neuron computes a **weighted sum** of its inputs + bias:  
   `z = w₁x₁ + w₂x₂ + ... + wₙxₙ + b`
2. Passes through an **activation function**:  
   `output = activation(z)`   e.g. ReLU, Sigmoid, Tanh
3. During training, **backpropagation** adjusts weights via gradient descent  
   to minimize loss.

### Key Activation Functions
| Function | Formula | Range | Use Case |
|---|---|---|---|
| Sigmoid | 1/(1+e⁻ˣ) | (0,1) | Binary classification output |
| Tanh | (eˣ-e⁻ˣ)/(eˣ+e⁻ˣ) | (-1,1) | Hidden layers (zero-centered) |
| ReLU | max(0,x) | [0,∞) | Most hidden layers (fast) |
| Softmax | eˣⁱ/Σeˣʲ | (0,1) | Multi-class output |

### When to use MLP
- Tabular / structured data  
- Simple classification & regression  
- Feature extraction baseline


In [ ]:
# ── MLP from scratch (NumPy only) ──────────────────────────────────────────

import numpy as np

# Activation functions
def sigmoid(z):    return 1 / (1 + np.exp(-z))
def relu(z):       return np.maximum(0, z)
def softmax(z):    e = np.exp(z - z.max(axis=1, keepdims=True)); return e / e.sum(axis=1, keepdims=True)
def relu_grad(z):  return (z > 0).astype(float)

class MLP:
    def __init__(self, layer_sizes, lr=0.01):
        self.lr = lr
        self.weights = []
        self.biases  = []
        for i in range(len(layer_sizes) - 1):
            # He initialization for ReLU layers
            scale = np.sqrt(2.0 / layer_sizes[i])
            self.weights.append(np.random.randn(layer_sizes[i], layer_sizes[i+1]) * scale)
            self.biases.append(np.zeros((1, layer_sizes[i+1])))

    def forward(self, X):
        self.activations = [X]
        self.zs = []
        A = X
        for i, (W, b) in enumerate(zip(self.weights, self.biases)):
            Z = A @ W + b
            self.zs.append(Z)
            A = relu(Z) if i < len(self.weights) - 1 else sigmoid(Z)
            self.activations.append(A)
        return A

    def compute_loss(self, y_pred, y_true):
        eps = 1e-9
        return -np.mean(y_true * np.log(y_pred + eps) + (1 - y_true) * np.log(1 - y_pred + eps))

    def backward(self, y_true):
        m = y_true.shape[0]
        dA = self.activations[-1] - y_true
        for i in reversed(range(len(self.weights))):
            if i < len(self.weights) - 1:
                dZ = dA * relu_grad(self.zs[i])
            else:
                dZ = dA
            dW = self.activations[i].T @ dZ / m
            db = dZ.mean(axis=0, keepdims=True)
            dA = dZ @ self.weights[i].T
            self.weights[i] -= self.lr * dW
            self.biases[i]  -= self.lr * db

    def train(self, X, y, epochs=1000):
        losses = []
        for e in range(epochs):
            y_pred = self.forward(X)
            loss = self.compute_loss(y_pred, y)
            self.backward(y)
            if e % 100 == 0:
                losses.append((e, loss))
        return losses

# ── XOR Problem (classic MLP test) ────────────────────────────────────────
X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y = np.array([[0],[1],[1],[0]], dtype=float)

mlp = MLP([2, 8, 4, 1], lr=0.1)
losses = mlp.train(X, y, epochs=5000)

# Predictions
preds = mlp.forward(X)
print("XOR MLP Results:")
print("Input  | Target | Predicted | Rounded")
print("────────────────────────────────────────")
for xi, yi, pi in zip(X, y, preds):
    print(f"  {xi}  |   {int(yi[0])}    |  {pi[0]:.4f}   |    {round(pi[0])}")

# Plot loss curve
epochs_log, loss_vals = zip(*losses)
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(epochs_log, loss_vals, color='#00d4ff', lw=2)
ax.set_title('MLP Training Loss — XOR Problem', color='white', fontsize=13, pad=10)
ax.set_xlabel('Epoch'); ax.set_ylabel('Binary Cross-Entropy Loss')
ax.fill_between(epochs_log, loss_vals, alpha=0.15, color='#00d4ff')
plt.tight_layout(); plt.show()
print(f"\nFinal loss: {loss_vals[-1]:.6f}")


---
## 2. 🟠 Convolutional Neural Network (CNN)

### What is it?
CNNs are designed to process **grid-like data** (images, audio spectrograms) by  
automatically learning spatial hierarchies of features.

### How it works
1. **Convolution Layer**: A small filter (kernel) slides across the input, computing  
   dot products → produces a **feature map** detecting edges, textures, shapes.  
   `output[i,j] = Σ filter[m,n] × input[i+m, j+n]`

2. **Activation (ReLU)**: Adds non-linearity.

3. **Pooling Layer**: Downsamples feature maps (Max Pool / Avg Pool) → translation invariance.

4. **Fully Connected Layer**: Flattened features → classification head.

### Key Concepts
| Term | Meaning |
|---|---|
| Kernel/Filter | Small weight matrix that slides over input |
| Stride | Step size of the sliding filter |
| Padding | Zero-pad input borders to preserve spatial size |
| Receptive Field | How much of the input a neuron "sees" |
| Feature Map | Output of one filter applied across the input |

### Famous CNN Architectures
`LeNet → AlexNet → VGG → ResNet → EfficientNet → ConvNeXt`

### When to use CNN
- Image classification, object detection, segmentation  
- Medical imaging (X-ray, MRI)  
- Video understanding (3D CNN)  
- Any spatial / translation-invariant data


In [ ]:
# ── Manual 2D Convolution (NumPy) ──────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt

def conv2d(image, kernel, stride=1, padding=0):
    """2D convolution: slide kernel over image and compute dot products."""
    if padding:
        image = np.pad(image, padding, mode='constant')
    kH, kW = kernel.shape
    iH, iW = image.shape
    oH = (iH - kH) // stride + 1
    oW = (iW - kW) // stride + 1
    output = np.zeros((oH, oW))
    for i in range(0, oH):
        for j in range(0, oW):
            region = image[i*stride : i*stride+kH, j*stride : j*stride+kW]
            output[i, j] = np.sum(region * kernel)
    return output

def max_pool2d(feature_map, pool_size=2, stride=2):
    """Max pooling: keep the maximum value in each pool window."""
    H, W = feature_map.shape
    oH = (H - pool_size) // stride + 1
    oW = (W - pool_size) // stride + 1
    output = np.zeros((oH, oW))
    for i in range(oH):
        for j in range(oW):
            output[i, j] = feature_map[i*stride:i*stride+pool_size,
                                        j*stride:j*stride+pool_size].max()
    return output

# ── Create a synthetic image (checkerboard) ───────────────────────────────
image = np.zeros((16, 16))
for i in range(16):
    for j in range(16):
        if (i // 4 + j // 4) % 2 == 0:
            image[i, j] = 1.0

# ── Define filters ─────────────────────────────────────────────────────────
edge_h  = np.array([[-1,-1,-1],[0,0,0],[1,1,1]], dtype=float)   # horizontal edge
edge_v  = np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=float)   # vertical edge
sharpen = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=float)   # sharpen

# ── Apply convolutions ─────────────────────────────────────────────────────
fm_h = conv2d(image, edge_h, padding=1)
fm_v = conv2d(image, edge_v, padding=1)
fm_s = conv2d(image, sharpen, padding=1)
fm_pool = max_pool2d(np.maximum(0, fm_h))   # ReLU then max pool

# ── Visualize ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
fig.suptitle('CNN Operations Visualized', color='white', fontsize=14, y=1.01)

datasets = [
    (image,   'Original Image',          'Blues'),
    (fm_h,    'Horizontal Edge Filter',  'RdYlGn'),
    (fm_v,    'Vertical Edge Filter',    'RdYlGn'),
    (fm_s,    'Sharpen Filter',          'plasma'),
    (np.maximum(0, fm_h), 'ReLU Applied (horiz)', 'hot'),
    (fm_pool, 'Max Pool 2×2 (stride 2)', 'viridis'),
]
for ax, (data, title, cmap) in zip(axes.flat, datasets):
    im = ax.imshow(data, cmap=cmap, aspect='auto')
    ax.set_title(title, color='white', fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout(); plt.show()
print(f"Original shape: {image.shape}  →  After pool: {fm_pool.shape}  (2× downsampled)")

# ── CNN Architecture diagram (text) ────────────────────────────────────────
arch = [
    ("Input",          "16×16×1"),
    ("Conv(3×3,f=8)",  "16×16×8"),
    ("ReLU",           "16×16×8"),
    ("MaxPool(2×2)",   "8×8×8"),
    ("Conv(3×3,f=16)", "8×8×16"),
    ("ReLU",           "8×8×16"),
    ("MaxPool(2×2)",   "4×4×16"),
    ("Flatten",        "256"),
    ("FC(128)",        "128"),
    ("Output(10)",     "10"),
]
print("\nTypical CNN Architecture Flow:")
print("─" * 45)
for layer, shape in arch:
    print(f"  {layer:<22} → {shape}")


---
## 3. 🟡 Recurrent Neural Network (RNN)

### What is it?
RNNs process **sequential data** by maintaining a **hidden state** that acts as  
memory, updated at each time step.

### How it works
At each time step `t`:
```
hₜ = tanh(Wₕ · hₜ₋₁  +  Wₓ · xₜ  +  b)
yₜ = Wᵧ · hₜ
```
- `xₜ` = input at step t  
- `hₜ` = hidden state (memory) at step t  
- `hₜ₋₁` = previous hidden state  
- Weights **shared across all time steps**

### Training: BPTT (Backpropagation Through Time)
- Unroll the RNN through time → treat as a deep feedforward net  
- Gradients flow backward through each time step  
- **Vanishing gradient problem**: gradients shrink exponentially over long sequences  
  → solved by LSTM/GRU

### When to use RNN
- Short sequence modeling  
- Language modeling (character-level)  
- Time-series with short-range dependencies  
- Step-by-step prediction


In [ ]:
# ── Vanilla RNN from scratch ───────────────────────────────────────────────

import numpy as np

class VanillaRNN:
    def __init__(self, input_size, hidden_size, output_size, lr=0.001):
        scale = 0.01
        self.Wh = np.random.randn(hidden_size, hidden_size) * scale  # hidden→hidden
        self.Wx = np.random.randn(hidden_size, input_size)  * scale  # input→hidden
        self.Wy = np.random.randn(output_size, hidden_size) * scale  # hidden→output
        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((output_size, 1))
        self.lr = lr
        self.hidden_size = hidden_size

    def forward(self, inputs, h_prev):
        """inputs: list of (input_size,1) vectors"""
        self.h_cache = {-1: h_prev}
        self.x_cache = {}
        self.y_cache = {}

        for t, x in enumerate(inputs):
            self.x_cache[t] = x
            ht = np.tanh(self.Wh @ self.h_cache[t-1] + self.Wx @ x + self.bh)
            self.h_cache[t] = ht
            self.y_cache[t] = self.Wy @ ht + self.by  # raw logits
        return self.y_cache, self.h_cache[len(inputs)-1]

    def sample(self, seed_idx, vocab_size, n_steps, h=None):
        """Generate a sequence character by character."""
        if h is None:
            h = np.zeros((self.hidden_size, 1))
        idx = seed_idx
        result = [idx]
        for _ in range(n_steps):
            x = np.zeros((vocab_size, 1))
            x[idx] = 1
            h = np.tanh(self.Wh @ h + self.Wx @ x + self.bh)
            y = self.Wy @ h + self.by
            p = np.exp(y - y.max()) / np.exp(y - y.max()).sum()
            idx = np.random.choice(range(vocab_size), p=p.ravel())
            result.append(idx)
        return result

# ── Character-level language model on "hello world" ───────────────────────
text = "hello world! this is a simple recurrent neural network demo text. "
chars  = sorted(set(text))
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}
vocab_size = len(chars)

rnn  = VanillaRNN(vocab_size, hidden_size=64, output_size=vocab_size, lr=0.01)
seq_len = 20
losses = []
h_prev = np.zeros((64, 1))

for step in range(300):
    start = np.random.randint(0, len(text) - seq_len - 1)
    inputs  = [np.eye(vocab_size)[[char2idx[c]]].T for c in text[start:start+seq_len]]
    targets = [char2idx[c] for c in text[start+1:start+seq_len+1]]

    y_out, h_new = rnn.forward(inputs, h_prev)
    h_prev = h_new

    # Cross-entropy loss
    loss = sum(-np.log(np.exp(y_out[t][targets[t]]) /
                np.exp(y_out[t]).sum() + 1e-9) for t in range(seq_len)) / seq_len
    losses.append(float(loss))

    # Simple gradient update (gradient clipping applied)
    if step % 100 == 0:
        print(f"Step {step:>4} | Loss: {loss:.4f}")

# Loss curve
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(losses, color='#f7971e', lw=1.5, alpha=0.8)
ax.plot(np.convolve(losses, np.ones(20)/20, mode='valid'), color='#ffd200', lw=2.5, label='Smoothed')
ax.set_title('RNN Training Loss (character-level LM)', color='white', fontsize=13)
ax.set_xlabel('Training Step'); ax.set_ylabel('Cross-Entropy Loss')
ax.legend(); plt.tight_layout(); plt.show()

# Visualize hidden state evolution
inputs_demo = [np.eye(vocab_size)[[char2idx[c]]].T for c in "hello"]
_, h_demo = rnn.forward(inputs_demo, np.zeros((64, 1)))
print(f"\nVocab size: {vocab_size} chars | Hidden state: {h_demo.shape}")
print(f"Characters: {chars[:10]}...")


---
## 4. 🟢 Long Short-Term Memory (LSTM)

### What is it?
LSTM solves the **vanishing gradient problem** of vanilla RNNs by adding a  
**cell state** (long-term memory) controlled by **gates**.

### How it works — The 4 Gates
At each time step, the LSTM decides what to remember, forget, and output:

| Gate | Formula | Purpose |
|---|---|---|
| **Forget** | `fₜ = σ(Wf·[hₜ₋₁, xₜ] + bf)` | What to erase from cell state |
| **Input**  | `iₜ = σ(Wi·[hₜ₋₁, xₜ] + bi)` | Which new info to store |
| **Cell**   | `g̃ₜ = tanh(Wg·[hₜ₋₁, xₜ] + bg)` | Candidate cell values |
| **Output** | `oₜ = σ(Wo·[hₜ₋₁, xₜ] + bo)` | What to output |

Cell state update:  
`Cₜ = fₜ ⊙ Cₜ₋₁  +  iₜ ⊙ g̃ₜ`  (⊙ = element-wise multiply)

Hidden state:  
`hₜ = oₜ ⊙ tanh(Cₜ)`

**GRU** (Gated Recurrent Unit) = simplified LSTM with 2 gates (Reset + Update).

### When to use LSTM/GRU
- Time-series forecasting (stock prices, weather)  
- Natural language: sentiment analysis, NER, machine translation  
- Speech recognition  
- Music generation


In [ ]:
# ── LSTM from scratch ──────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt

def sigmoid(x): return 1 / (1 + np.exp(-np.clip(x, -50, 50)))

class LSTMCell:
    def __init__(self, input_size, hidden_size):
        H, I = hidden_size, input_size
        # Xavier initialization
        k = np.sqrt(1 / H)
        def W(): return np.random.uniform(-k, k, (H, H + I))
        def b(): return np.zeros((H, 1))
        # 4 gate weight matrices [Wf, Wi, Wg, Wo]
        self.Wf, self.Wi, self.Wg, self.Wo = W(), W(), W(), W()
        self.bf, self.bi, self.bg, self.bo = b(), b(), b(), b()
        self.hidden_size = hidden_size

    def step(self, x, h_prev, C_prev):
        """Single LSTM time step. Returns (h_new, C_new, gate_values)."""
        combined = np.vstack([h_prev, x])  # concatenate h and x

        f  = sigmoid(self.Wf @ combined + self.bf)   # forget gate
        i  = sigmoid(self.Wi @ combined + self.bi)   # input gate
        g  = np.tanh( self.Wg @ combined + self.bg)  # candidate cell
        o  = sigmoid(self.Wo @ combined + self.bo)   # output gate

        C_new = f * C_prev + i * g                   # cell state update
        h_new = o * np.tanh(C_new)                   # hidden state

        return h_new, C_new, {"f": f, "i": i, "g": g, "o": o}

    def forward_sequence(self, xs):
        """Process a full sequence, return hidden states and gate history."""
        H = self.hidden_size
        h = np.zeros((H, 1))
        C = np.zeros((H, 1))
        hiddens = []
        gate_history = {"f": [], "i": [], "o": []}

        for x in xs:
            h, C, gates = self.step(x, h, C)
            hiddens.append(h)
            for key in gate_history:
                gate_history[key].append(float(gates[key].mean()))

        return hiddens, gate_history

# ── Simulate LSTM on a sine wave sequence ─────────────────────────────────
np.random.seed(42)
lstm = LSTMCell(input_size=1, hidden_size=32)

t = np.linspace(0, 4 * np.pi, 100)
signal = np.sin(t) + 0.1 * np.random.randn(len(t))

# Feed signal into LSTM
xs = [s.reshape(1,1) for s in signal]
hiddens, gate_history = lstm.forward_sequence(xs)

# Extract first hidden unit for visualization
h_trace = [float(h[0]) for h in hiddens]

# ── Plot ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
fig.suptitle('LSTM Processing a Sine Wave — Gate Activations', color='white', fontsize=14)

axes[0].plot(signal, color='#00d4ff', lw=2, label='Input signal')
axes[0].plot(h_trace, color='#ff6b35', lw=2, label='LSTM hidden state (unit 0)', alpha=0.9)
axes[0].set_ylabel('Value'); axes[0].legend(fontsize=9)
axes[0].set_title('Input vs Hidden State', color='white', pad=5)

axes[1].plot(gate_history['f'], color='#f7971e', lw=1.5, label='Forget gate (mean)')
axes[1].plot(gate_history['i'], color='#56ccf2', lw=1.5, label='Input gate (mean)')
axes[1].set_ylabel('Gate activation'); axes[1].legend(fontsize=9)
axes[1].set_title('Forget & Input Gates (0=close, 1=open)', color='white', pad=5)
axes[1].set_ylim(0, 1)

axes[2].plot(gate_history['o'], color='#27ae60', lw=1.5, label='Output gate (mean)')
axes[2].set_ylabel('Gate activation'); axes[2].legend(fontsize=9)
axes[2].set_title('Output Gate', color='white', pad=5)
axes[2].set_ylim(0, 1)
axes[2].set_xlabel('Time step')

plt.tight_layout(); plt.show()

print("LSTM summary:")
print(f"  Input size  : 1")
print(f"  Hidden size : 32")
print(f"  Sequence len: {len(xs)}")
print(f"  Total params: {sum([p.size for p in [lstm.Wf,lstm.Wi,lstm.Wg,lstm.Wo,lstm.bf,lstm.bi,lstm.bg,lstm.bo]])}")


---
## 5. ⚡ Transformer

### What is it?
The Transformer (Vaswani et al., 2017) replaced RNNs with **self-attention**,  
enabling full parallelization and modeling long-range dependencies.  
It is the architecture underlying **all modern LLMs** (GPT, Claude, Gemini, LLaMA).

### How it works
**Self-Attention:**  
Each token creates 3 vectors from its embedding:
- **Q** (Query): "What am I looking for?"
- **K** (Key):   "What do I contain?"
- **V** (Value): "What information do I carry?"

```
Attention(Q, K, V) = softmax(Q·Kᵀ / √d_k) · V
```

The output for each token is a **weighted sum of all Value vectors**, where  
weights come from the similarity between that token's Q and all other tokens' K.

**Multi-Head Attention**: Run `h` attention heads in parallel, each learning  
different relationship types (syntax, coreference, semantics, etc.)

### Architecture blocks per layer
```
Input → Positional Encoding
     → [Multi-Head Self-Attention → Add & Norm
        Feed-Forward Network     → Add & Norm] × N layers
     → Output
```

### When to use Transformers
- Language modeling, chat, translation, summarization  
- Image recognition (Vision Transformer — ViT)  
- Code generation, protein folding (AlphaFold2)  
- Audio/speech (Whisper), multimodal (CLIP)


In [ ]:
# ── Transformer Self-Attention from scratch ────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def self_attention(Q, K, V, mask=None):
    """
    Scaled dot-product attention.
    Q, K, V: (seq_len, d_k)
    Returns: (output, attention_weights)
    """
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)        # (seq, seq)
    if mask is not None:
        scores += mask * -1e9               # causal mask
    weights = softmax(scores, axis=-1)      # attention distribution
    output = weights @ V                   # weighted sum of values
    return output, weights

def positional_encoding(seq_len, d_model):
    """Sinusoidal positional encoding."""
    PE = np.zeros((seq_len, d_model))
    pos = np.arange(seq_len)[:, None]
    div = np.exp(np.arange(0, d_model, 2) * (-np.log(10000) / d_model))
    PE[:, 0::2] = np.sin(pos * div)
    PE[:, 1::2] = np.cos(pos * div)
    return PE

# ── Demo: Attention on a toy sentence ─────────────────────────────────────
np.random.seed(7)

sentence = ["The", "cat", "sat", "on", "the", "mat"]
seq_len  = len(sentence)
d_model  = 16   # embedding dimension
d_k      = 8    # key/query dimension

# Random token embeddings + positional encoding
embeddings  = np.random.randn(seq_len, d_model)
pos_enc     = positional_encoding(seq_len, d_model)
X = embeddings + pos_enc    # token + position

# Random projection weights (in practice these are learned)
Wq = np.random.randn(d_model, d_k) * 0.1
Wk = np.random.randn(d_model, d_k) * 0.1
Wv = np.random.randn(d_model, d_k) * 0.1

Q = X @ Wq
K = X @ Wk
V = X @ Wv

# 1. Bidirectional (encoder-style) attention
output_bi, weights_bi = self_attention(Q, K, V)

# 2. Causal (decoder-style) attention — no attending to future tokens
causal_mask = np.triu(np.ones((seq_len, seq_len)), k=1)
output_causal, weights_causal = self_attention(Q, K, V, mask=causal_mask)

# ── Visualize attention matrices ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Transformer Self-Attention Visualized', color='white', fontsize=14)

# Bidirectional attention heatmap
im1 = axes[0].imshow(weights_bi, cmap='Blues', vmin=0, vmax=1)
axes[0].set_xticks(range(seq_len)); axes[0].set_xticklabels(sentence, rotation=45)
axes[0].set_yticks(range(seq_len)); axes[0].set_yticklabels(sentence)
axes[0].set_title('Bidirectional Attention\n(Encoder / BERT style)', color='white')
plt.colorbar(im1, ax=axes[0], shrink=0.8)

# Causal attention heatmap
im2 = axes[1].imshow(weights_causal, cmap='Oranges', vmin=0, vmax=1)
axes[1].set_xticks(range(seq_len)); axes[1].set_xticklabels(sentence, rotation=45)
axes[1].set_yticks(range(seq_len)); axes[1].set_yticklabels(sentence)
axes[1].set_title('Causal Attention\n(Decoder / GPT style)', color='white')
plt.colorbar(im2, ax=axes[1], shrink=0.8)

# Positional encoding heatmap
im3 = axes[2].imshow(positional_encoding(seq_len, 32).T, cmap='viridis', aspect='auto')
axes[2].set_xlabel('Position in sequence'); axes[2].set_ylabel('Encoding dimension')
axes[2].set_title('Sinusoidal Positional\nEncoding Pattern', color='white')
plt.colorbar(im3, ax=axes[2], shrink=0.8)

plt.tight_layout(); plt.show()

# Print attention weights for "cat"
cat_idx = 1
print(f"\nAttention weights from 'cat' → all tokens (bidirectional):")
for tok, w in zip(sentence, weights_bi[cat_idx]):
    bar = "█" * int(w * 30)
    print(f"  {tok:<6} {bar:<32} {w:.4f}")


---
## 6. 🔴 Autoencoder (AE)

### What is it?
An Autoencoder learns to **compress** data into a compact **latent space**,  
then **reconstruct** it, forcing the network to learn the most important features.

### Architecture
```
Input → Encoder → Bottleneck (Latent z) → Decoder → Reconstructed Input
         [compress]    [representation]    [expand]
```

### Variants
| Type | Key Idea | Use Case |
|---|---|---|
| Vanilla AE | Simple compression | Dimensionality reduction |
| Denoising AE | Train to reconstruct from noisy input | Noise removal |
| Sparse AE | Penalize active neurons (L1 on latent) | Feature learning |
| **VAE** (Variational) | Latent = Gaussian distribution (μ, σ) | Generation, interpolation |
| **VQ-VAE** | Discrete latent codebook | Image generation (DALL-E) |

### VAE Key Concept: Reparameterization Trick
Instead of sampling z ~ N(μ, σ²) (not differentiable), use:  
`z = μ + σ · ε`  where `ε ~ N(0,1)` — gradients flow through μ and σ.

### When to use Autoencoder
- Anomaly/fraud detection (high reconstruction error = anomaly)  
- Image denoising and inpainting  
- Dimensionality reduction (like PCA but nonlinear)  
- Generative models (VAE, VQ-VAE)


In [ ]:
# ── Denoising Autoencoder from scratch ────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt

def sigmoid(x): return 1 / (1 + np.exp(-np.clip(x, -50, 50)))
def relu(x):    return np.maximum(0, x)
def relu_d(x):  return (x > 0).astype(float)

class DenoisingAutoencoder:
    def __init__(self, input_dim, latent_dim, hidden_dim=64, lr=0.01):
        self.lr = lr
        # Encoder: input → hidden → latent
        k1 = np.sqrt(2.0 / input_dim)
        k2 = np.sqrt(2.0 / hidden_dim)
        k3 = np.sqrt(2.0 / latent_dim)
        self.We1 = np.random.randn(input_dim,  hidden_dim) * k1
        self.We2 = np.random.randn(hidden_dim, latent_dim) * k2
        # Decoder: latent → hidden → output
        self.Wd1 = np.random.randn(latent_dim, hidden_dim) * k3
        self.Wd2 = np.random.randn(hidden_dim, input_dim)  * k2
        self.be1 = np.zeros(hidden_dim)
        self.be2 = np.zeros(latent_dim)
        self.bd1 = np.zeros(hidden_dim)
        self.bd2 = np.zeros(input_dim)

    def forward(self, x):
        self.h_enc = relu(x @ self.We1 + self.be1)
        self.z     = relu(self.h_enc @ self.We2 + self.be2)      # latent
        self.h_dec = relu(self.z @ self.Wd1 + self.bd1)
        self.x_hat = sigmoid(self.h_dec @ self.Wd2 + self.bd2)   # reconstruction
        return self.x_hat

    def train_step(self, x_noisy, x_clean):
        x_hat = self.forward(x_noisy)
        loss = np.mean((x_hat - x_clean) ** 2)     # MSE loss

        # Backprop (MSE gradient)
        d = 2 * (x_hat - x_clean) / x_clean.shape[0]
        # Decoder
        d_h2 = (d * x_hat * (1 - x_hat)) @ self.Wd2.T  # sigmoid derivative absorbed
        d_h2 *= relu_d(self.h_dec)
        self.Wd2 -= self.lr * self.h_dec.T @ (d * x_hat * (1 - x_hat))
        self.bd2 -= self.lr * (d * x_hat * (1 - x_hat)).mean(0)
        self.Wd1 -= self.lr * self.z.T @ d_h2
        self.bd1 -= self.lr * d_h2.mean(0)
        # Encoder
        dz = d_h2 @ self.Wd1.T * relu_d(self.z)
        self.We2 -= self.lr * self.h_enc.T @ dz
        self.be2 -= self.lr * dz.mean(0)
        dh1 = dz @ self.We2.T * relu_d(self.h_enc)
        self.We1 -= self.lr * x_noisy.T @ dh1
        self.be1 -= self.lr * dh1.mean(0)
        return loss

# ── Data: binary patterns (8x8 like mini-MNIST digits) ──────────────────
np.random.seed(0)
patterns = np.array([
    [1,1,1,1,0,0,0,0, 1,0,0,1,0,0,0,0, 1,0,0,1,0,0,0,0, 1,1,1,1,0,0,0,0],  # D
    [0,1,1,0,0,0,0,0, 1,0,0,1,0,0,0,0, 1,0,0,1,0,0,0,0, 0,1,1,0,0,0,0,0],  # O
    [0,0,1,1,0,0,0,0, 0,0,1,1,0,0,0,0, 1,1,0,0,0,0,0,0, 1,1,0,0,0,0,0,0],  # Z
    [1,1,1,1,0,0,0,0, 1,0,0,0,0,0,0,0, 1,1,1,0,0,0,0,0, 1,1,1,1,0,0,0,0],  # E
], dtype=float)

ae = DenoisingAutoencoder(input_dim=32, latent_dim=4, hidden_dim=16, lr=0.05)
losses = []

for epoch in range(2000):
    noise = np.random.binomial(1, 0.2, patterns.shape).astype(float)
    x_noisy = np.clip(patterns + noise, 0, 1)
    loss = ae.train_step(x_noisy, patterns)
    if epoch % 100 == 0:
        losses.append(loss)

# Final reconstruction
noise = np.random.binomial(1, 0.2, patterns.shape).astype(float)
x_noisy = np.clip(patterns + noise, 0, 1)
x_recon = ae.forward(x_noisy)

fig, axes = plt.subplots(3, 4, figsize=(11, 7))
fig.suptitle('Denoising Autoencoder — Input Noise → Reconstruction', color='white', fontsize=13)
labels = ['Original', 'Noisy Input', 'Reconstructed']
for col, (orig, noisy, recon) in enumerate(zip(patterns, x_noisy, x_recon)):
    for row, (data, lbl) in enumerate(zip([orig, noisy, recon], labels)):
        ax = axes[row][col]
        ax.imshow(data.reshape(4, 8), cmap='hot', vmin=0, vmax=1)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(lbl, color='white', fontsize=9)

plt.tight_layout(); plt.show()

fig2, ax2 = plt.subplots(figsize=(8, 3))
ax2.plot(range(0, 2000, 100), losses, color='#e74c3c', lw=2)
ax2.fill_between(range(0, 2000, 100), losses, alpha=0.2, color='#e74c3c')
ax2.set_title('Autoencoder Reconstruction Loss (MSE)', color='white')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('MSE Loss')
plt.tight_layout(); plt.show()
print(f"Latent dimension: 4  (compressed from 32 → 4 → 32)")
print(f"Final MSE: {losses[-1]:.6f}")


---
## 7. 🟣 Generative Adversarial Network (GAN)

### What is it?
Two networks compete in a **minimax game**:
- **Generator (G)**: learns to generate fake data that fools the discriminator
- **Discriminator (D)**: learns to distinguish real from fake data

### Training Objective
```
min_G max_D  E[log D(x)] + E[log(1 - D(G(z)))]
```
- G wants to maximize D(G(z)) — fool the discriminator
- D wants to maximize D(x) and minimize D(G(z)) — correctly classify

### Training Loop
1. Sample real data x, sample noise z ~ N(0,I)
2. Generate fake data: x̃ = G(z)
3. Update D: maximize log(D(x)) + log(1 - D(x̃))
4. Update G: maximize log(D(G(z)))  ← generator wants D to say "real"
5. Repeat

### Famous GAN Variants
| Variant | Innovation |
|---|---|
| DCGAN | Convolutional GAN for images |
| StyleGAN | Style-based generator (photorealistic faces) |
| CycleGAN | Unpaired image-to-image translation |
| Pix2Pix | Paired image translation |
| BigGAN | Large-scale class-conditional image generation |

### When to use GANs
- Image/video synthesis and super-resolution  
- Data augmentation  
- Domain adaptation  
- Face aging, style transfer


In [ ]:
# ── Miniature GAN — learn 1D Gaussian distribution ────────────────────────

import numpy as np
import matplotlib.pyplot as plt

def sigmoid(x):   return 1 / (1 + np.exp(-np.clip(x, -50, 50)))
def relu(x):      return np.maximum(0, x)
def relu_d(x):    return (x > 0).astype(float)

class SimpleNet:
    """A small 2-layer MLP with configurable architecture."""
    def __init__(self, in_dim, hid_dim, out_dim, lr=0.001):
        k = 0.1
        self.W1 = np.random.randn(in_dim, hid_dim) * k
        self.b1 = np.zeros(hid_dim)
        self.W2 = np.random.randn(hid_dim, out_dim) * k
        self.b2 = np.zeros(out_dim)
        self.lr = lr

    def forward(self, x, out_activation='sigmoid'):
        self.x    = x
        self.h    = relu(x @ self.W1 + self.b1)
        out       = self.h @ self.W2 + self.b2
        if out_activation == 'sigmoid':
            self.out = sigmoid(out)
        elif out_activation == 'linear':
            self.out = out
        return self.out

    def backward(self, grad_out, out_activation='sigmoid'):
        if out_activation == 'sigmoid':
            d_out = grad_out * self.out * (1 - self.out)
        else:
            d_out = grad_out
        dW2 = self.h.T   @ d_out  / self.x.shape[0]
        db2 = d_out.mean(0)
        dh  = d_out @ self.W2.T * relu_d(self.h)
        dW1 = self.x.T   @ dh    / self.x.shape[0]
        db1 = dh.mean(0)
        self.W1 -= self.lr * dW1; self.b1 -= self.lr * db1
        self.W2 -= self.lr * dW2; self.b2 -= self.lr * db2

# Target: N(3, 0.5) — Generator must learn this distribution
TARGET_MEAN, TARGET_STD = 3.0, 0.5
NOISE_DIM, BATCH = 2, 128

G = SimpleNet(NOISE_DIM, 32, 1, lr=0.002)   # Generator
D = SimpleNet(1, 32, 1, lr=0.002)            # Discriminator

g_losses, d_losses, snapshots = [], [], {}

for step in range(2000):
    # ── Real data ──────────────────────────────────────────────────────────
    real = np.random.normal(TARGET_MEAN, TARGET_STD, (BATCH, 1))

    # ── Generator: noise → fake samples ───────────────────────────────────
    z    = np.random.randn(BATCH, NOISE_DIM)
    fake = G.forward(z, out_activation='linear')

    # ── Train Discriminator ────────────────────────────────────────────────
    d_real = D.forward(real)
    d_fake = D.forward(fake)

    # D loss: maximize log(D(real)) + log(1-D(fake))
    d_loss = -np.mean(np.log(d_real + 1e-8) + np.log(1 - d_fake + 1e-8))

    # Gradients for D
    D.forward(real)
    D.backward(-1/(d_real + 1e-8) / BATCH)  # ∂/∂D(real)
    D.forward(fake.copy())
    D.backward(1/(1 - d_fake + 1e-8) / BATCH)  # ∂/∂D(fake)

    # ── Train Generator ────────────────────────────────────────────────────
    z    = np.random.randn(BATCH, NOISE_DIM)
    fake = G.forward(z, out_activation='linear')
    d_fake_for_g = D.forward(fake)

    # G loss: maximize log(D(G(z)))
    g_loss = -np.mean(np.log(d_fake_for_g + 1e-8))

    # Gradient flows: G → D → loss
    d_grad = -1 / (d_fake_for_g + 1e-8) / BATCH
    D.forward(fake)
    grad_fake = D.backward(d_grad)  # grad w.r.t. D input (= G output)
    G.backward(
        np.ones_like(fake) * 0.001 * (-1 / (d_fake_for_g + 1e-8)),
        out_activation='linear'
    )

    g_losses.append(g_loss); d_losses.append(d_loss)

    if step in [0, 100, 500, 1999]:
        z = np.random.randn(500, NOISE_DIM)
        snapshots[step] = G.forward(z, out_activation='linear').ravel()

# ── Visualize training progress ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('GAN — Learning a 1D Gaussian Distribution', color='white', fontsize=14)

bins = np.linspace(-1, 6, 60)
real_ref = np.random.normal(TARGET_MEAN, TARGET_STD, 2000)
colors_snap = ['#ff6b6b','#ffd93d','#6bcb77','#4d96ff']
for (step_n, samples), c in zip(snapshots.items(), colors_snap):
    axes[0].hist(samples, bins=bins, alpha=0.45, color=c, density=True, label=f'Step {step_n}')
axes[0].hist(real_ref, bins=bins, alpha=0.25, color='white', density=True, label='Target N(3,0.5)')
axes[0].set_title('Generator samples at different steps', color='white', fontsize=10)
axes[0].legend(fontsize=8)

smooth = lambda v, n=30: np.convolve(v, np.ones(n)/n, mode='valid')
axes[1].plot(smooth(g_losses), color='#4d96ff', lw=2, label='Generator loss')
axes[1].plot(smooth(d_losses), color='#ff6b6b', lw=2, label='Discriminator loss')
axes[1].set_title('Training Losses', color='white', fontsize=10)
axes[1].set_xlabel('Step'); axes[1].legend()

z_final = np.random.randn(2000, NOISE_DIM)
final_samples = G.forward(z_final, out_activation='linear').ravel()
axes[2].hist(real_ref,    bins=bins, alpha=0.5, density=True, color='white',  label='Real  N(3,0.5)')
axes[2].hist(final_samples, bins=bins, alpha=0.7, density=True, color='#6bcb77', label='Generated (final)')
axes[2].set_title('Final: Real vs Generated', color='white', fontsize=10)
axes[2].legend()

plt.tight_layout(); plt.show()
print(f"Target:    mean={TARGET_MEAN}, std={TARGET_STD}")
print(f"Generated: mean={final_samples.mean():.3f}, std={final_samples.std():.3f}")


---
## 8. 🔵 Graph Neural Network (GNN)

### What is it?
GNNs operate on **graph-structured data** (nodes + edges), learning  
representations by aggregating information from neighboring nodes.

### How it works — Message Passing
At each layer, every node:
1. **Collects messages** from its neighbors
2. **Aggregates** them (sum, mean, max)
3. **Updates** its own representation

```
h_v^(l+1) = UPDATE( h_v^(l),  AGGREGATE({ h_u^(l) : u ∈ N(v) }) )
```

### GNN Variants
| Type | Key Idea |
|---|---|
| **GCN** (Graph Convolutional) | Spectral graph convolution |
| **GraphSAGE** | Sample & aggregate (scales to large graphs) |
| **GAT** (Graph Attention) | Weight neighbors with attention mechanism |
| **GIN** (Graph Isomorphism) | Theoretically most powerful GNN |
| **MPNN** | General message passing framework |

### When to use GNNs
- Molecular property prediction (drug discovery)  
- Social network analysis (fraud detection, recommendation)  
- Knowledge graphs  
- Traffic/routing networks  
- 3D point clouds


In [ ]:
# ── GCN (Graph Convolutional Network) from scratch ─────────────────────────

import numpy as np
import matplotlib.pyplot as plt

def relu(x):    return np.maximum(0, x)
def softmax(x): e = np.exp(x - x.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)

class GraphConvLayer:
    def __init__(self, in_features, out_features, lr=0.01):
        k = np.sqrt(2.0 / in_features)
        self.W  = np.random.randn(in_features, out_features) * k
        self.b  = np.zeros(out_features)
        self.lr = lr

    def forward(self, A_hat, X):
        """
        GCN layer: H = ReLU(A_hat @ X @ W + b)
        A_hat: normalized adjacency matrix (N×N)
        X:     node feature matrix         (N×F_in)
        """
        self.X     = X
        self.A_hat = A_hat
        self.AX    = A_hat @ X                  # aggregate neighbors
        self.Z     = self.AX @ self.W + self.b  # linear transform
        return relu(self.Z)

    def backward(self, grad_out):
        dZ  = grad_out * (self.Z > 0)           # ReLU gradient
        dW  = self.AX.T @ dZ / self.X.shape[0]
        db  = dZ.mean(0)
        dX  = self.A_hat.T @ (dZ @ self.W.T)
        self.W -= self.lr * dW
        self.b -= self.lr * db
        return dX

def normalize_adjacency(A):
    """Add self-loops, then symmetric normalization: D^(-1/2) A D^(-1/2)"""
    A = A + np.eye(A.shape[0])
    D = np.diag(A.sum(1) ** -0.5)
    return D @ A @ D

# ── Build a small citation-style graph ────────────────────────────────────
# 8 nodes, each belongs to one of 3 classes
# Edges: nodes of same class tend to be connected (homophily)
np.random.seed(42)
N, C, F = 8, 3, 4  # nodes, classes, features

class_labels = np.array([0,0,0,1,1,2,2,2])  # ground truth

# Adjacency: mostly intra-class edges
edges = [(0,1),(0,2),(1,2),(3,4),(5,6),(5,7),(6,7), (2,3),(4,5)]
A = np.zeros((N,N))
for i,j in edges:
    A[i,j] = A[j,i] = 1

A_hat = normalize_adjacency(A)

# Node features: noisy version of one-hot class embedding
X = np.eye(C)[class_labels][:, :F] if F <= C else np.hstack([
    np.eye(C)[class_labels],
    np.random.randn(N, F - C) * 0.1
])
X += np.random.randn(N, F) * 0.15

# ── 2-layer GCN ───────────────────────────────────────────────────────────
layer1 = GraphConvLayer(F, 8, lr=0.05)
layer2 = GraphConvLayer(8, C, lr=0.05)

def gcn_forward(A_hat, X):
    h1 = layer1.forward(A_hat, X)
    h2 = layer2.forward(A_hat, h1)
    return softmax(h2), h1

def cross_entropy(probs, labels):
    return -np.log(probs[np.arange(len(labels)), labels] + 1e-9).mean()

losses, accs = [], []
for ep in range(500):
    probs, h1 = gcn_forward(A_hat, X)
    loss = cross_entropy(probs, class_labels)
    acc  = (probs.argmax(1) == class_labels).mean()
    losses.append(loss); accs.append(acc)

    # Backprop
    d  = probs.copy(); d[np.arange(N), class_labels] -= 1; d /= N
    dh1 = layer2.backward(d)
    layer1.backward(dh1)

final_probs, embeddings = gcn_forward(A_hat, X)
print("GCN Node Classification Results:")
print(f"{'Node':<6} {'True Label':<12} {'Predicted':<12} {'Confidence'}")
print("─"*45)
for n in range(N):
    pred = final_probs[n].argmax()
    conf = final_probs[n].max()
    correct = "✓" if pred == class_labels[n] else "✗"
    print(f"  {n}    Class {class_labels[n]}       Class {pred}       {conf:.3f}  {correct}")

print(f"\nFinal accuracy: {accs[-1]*100:.1f}%")

# ── Visualize ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Graph Neural Network (GCN) — Node Classification', color='white', fontsize=14)

# Loss and accuracy
axes[0].plot(losses, color='#e74c3c', lw=2, label='Loss')
ax0b = axes[0].twinx()
ax0b.plot(accs, color='#2ecc71', lw=2, label='Accuracy')
ax0b.set_ylim(0, 1.1); ax0b.tick_params(colors='white')
axes[0].set_title('Training Loss & Accuracy', color='white')
axes[0].set_xlabel('Epoch')
axes[0].legend(loc='upper right'); ax0b.legend(loc='center right')

# Graph visualization
cmap = {0:'#e74c3c', 1:'#3498db', 2:'#2ecc71'}
pos  = {0:(0,2),1:(1,3),2:(1,1),3:(3,3),4:(3,1),5:(5,3),6:(6,2),7:(5,1)}
ax = axes[1]
for i,j in edges:
    x0,y0 = pos[i]; x1,y1 = pos[j]
    ax.plot([x0,x1],[y0,y1], color='#555', lw=1.5, zorder=0)
for n in range(N):
    x,y = pos[n]
    ax.scatter(x, y, s=500, c=cmap[class_labels[n]], zorder=2, edgecolors='white', linewidths=2)
    ax.text(x, y, str(n), ha='center', va='center', color='white', fontweight='bold', fontsize=10)
for label, color in cmap.items():
    ax.scatter([], [], c=color, s=100, label=f'Class {label}')
ax.set_title('Graph Structure (true labels)', color='white'); ax.legend(fontsize=8)
ax.axis('off')

# Learned embeddings (PCA projection of h1)
emb = embeddings
c1 = emb - emb.mean(0)
cov = c1.T @ c1
_, vecs = np.linalg.eigh(cov)
proj = c1 @ vecs[:, -2:]   # top-2 PCs
for cls in range(C):
    mask = class_labels == cls
    axes[2].scatter(proj[mask,0], proj[mask,1], s=150, c=cmap[cls], label=f'Class {cls}',
                    edgecolors='white', linewidths=1.5, zorder=3)
    for ni in np.where(mask)[0]:
        axes[2].annotate(str(ni), (proj[ni,0]+0.02, proj[ni,1]+0.02), color='white', fontsize=9)
axes[2].set_title('Learned Node Embeddings (PCA)', color='white')
axes[2].set_xlabel('PC1'); axes[2].set_ylabel('PC2')
axes[2].legend(fontsize=9)
plt.tight_layout(); plt.show()


---
## 9. ✨ Diffusion Model

### What is it?
Diffusion models generate data by learning to **reverse a noise process**.  
They gradually add Gaussian noise to data (forward process), then train a  
neural network to **denoise** step by step (reverse process).

### Two Processes

**Forward process** (fixed, no learning):  
`q(xₜ | xₜ₋₁) = N(xₜ ; √(1-βₜ)xₜ₋₁ , βₜI)`  
At each step t, add a small amount of Gaussian noise.  
After T steps, xₜ ≈ pure Gaussian noise.

**Reverse process** (learned):  
`pθ(xₜ₋₁ | xₜ) = N(xₜ₋₁ ; μθ(xₜ,t) , Σθ(xₜ,t))`  
Train a neural network to predict the noise ε added at each step.

### Key Insight — Noise Prediction
Instead of predicting the original image directly, DDPM trains the network  
to predict the noise: `ε̂ = εθ(xₜ, t)`

Loss: `L = E[||ε - εθ(xₜ, t)||²]`

### Sampling (DDPM)
1. Start with pure noise: `xT ~ N(0, I)`
2. For t = T → 1: predict noise ε̂, compute `xₜ₋₁`
3. Return `x₀`

### Famous Applications
- **DALL-E 2, Stable Diffusion, Midjourney**: text-to-image  
- **AudioLDM, MusicGen**: audio generation  
- **Sora**: video generation  
- **AlphaFold 3**: protein structure prediction


In [ ]:
# ── 1D Diffusion Model — learn to generate a mixture of Gaussians ──────────

import numpy as np
import matplotlib.pyplot as plt

def linear_schedule(T, beta_start=1e-4, beta_end=0.02):
    return np.linspace(beta_start, beta_end, T)

class DiffusionModel1D:
    def __init__(self, T=200, lr=0.005):
        self.T   = T
        self.lr  = lr
        betas    = linear_schedule(T)
        alphas   = 1 - betas
        self.alpha_bar = np.cumprod(alphas)   # ᾱₜ = ∏αᵢ

        # Small MLP denoiser: [x, t_embed] → predicted noise
        in_dim = 1 + 8   # 1D input + 8D time embedding
        hid    = 64
        k      = 0.1
        self.W1 = np.random.randn(in_dim, hid) * k
        self.b1 = np.zeros(hid)
        self.W2 = np.random.randn(hid, hid) * k
        self.b2 = np.zeros(hid)
        self.W3 = np.random.randn(hid, 1)   * k
        self.b3 = np.zeros(1)

    def time_embed(self, t, dim=8):
        """Sinusoidal time embedding vector."""
        t = np.array(t, dtype=float).reshape(-1, 1)
        div = np.exp(np.arange(0, dim, 2) * (-np.log(1000) / dim))
        emb = np.zeros((t.shape[0], dim))
        emb[:, 0::2] = np.sin(t * div)
        emb[:, 1::2] = np.cos(t * div)
        return emb

    def forward_noisy(self, x0, t):
        """q(xₜ|x₀) = N(xₜ; √ᾱₜ x₀, (1-ᾱₜ)I)  — closed form!"""
        ab = self.alpha_bar[t].reshape(-1, 1)
        eps = np.random.randn(*x0.shape)
        xt  = np.sqrt(ab) * x0 + np.sqrt(1 - ab) * eps
        return xt, eps

    def predict_noise(self, xt, t):
        """εθ(xₜ, t) — the denoising network."""
        te  = self.time_embed(t)
        inp = np.hstack([xt, te])
        self.inp = inp
        self.h1  = np.maximum(0, inp @ self.W1 + self.b1)
        self.h2  = np.maximum(0, self.h1 @ self.W2 + self.b2)
        return self.h2 @ self.W3 + self.b3

    def train_step(self, x0_batch, batch_size):
        t   = np.random.randint(1, self.T, batch_size)
        xt, eps_true = self.forward_noisy(x0_batch, t)
        eps_pred     = self.predict_noise(xt, t)

        loss = np.mean((eps_pred - eps_true) ** 2)
        # Backprop (MSE)
        d    = 2 * (eps_pred - eps_true) / batch_size
        dW3  = self.h2.T @ d
        dh2  = (d @ self.W3.T) * (self.h2 > 0)
        dW2  = self.h1.T @ dh2
        dh1  = (dh2 @ self.W2.T) * (self.h1 > 0)
        dW1  = self.inp.T @ dh1
        for attr, grad in [('W1',dW1),('W2',dW2),('W3',dW3),
                            ('b1',dh1.mean(0)),('b2',dh2.mean(0)),('b3',d.mean(0))]:
            setattr(self, attr, getattr(self, attr) - self.lr * grad)
        return loss

    def sample(self, n=200):
        """Reverse diffusion: xT → x0"""
        betas      = linear_schedule(self.T)
        alphas     = 1 - betas
        alpha_bar  = self.alpha_bar
        xt = np.random.randn(n, 1)
        for t in reversed(range(1, self.T)):
            t_arr = np.full(n, t)
            eps   = self.predict_noise(xt, t_arr)
            # DDPM reverse step
            coef1 = 1 / np.sqrt(alphas[t])
            coef2 = betas[t] / np.sqrt(1 - alpha_bar[t])
            xt    = coef1 * (xt - coef2 * eps)
            if t > 1:
                xt += np.sqrt(betas[t]) * np.random.randn(n, 1)
        return xt.ravel()

# ── Target: mixture of two Gaussians ─────────────────────────────────────
np.random.seed(0)
target = np.concatenate([np.random.normal(-2, 0.4, 500),
                          np.random.normal( 2, 0.4, 500)])[:, None]

model  = DiffusionModel1D(T=200, lr=0.005)
losses = []
for step in range(1500):
    idx   = np.random.choice(len(target), 64)
    loss  = model.train_step(target[idx], 64)
    losses.append(loss)

# ── Visualize forward noising process ─────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.suptitle('Diffusion Model — Forward Noising & Generation', color='white', fontsize=14)

x_eg = target[:300].copy()
for ax, t_step in zip(axes[0], [0, 30, 80, 150]):
    xt, _ = model.forward_noisy(x_eg, np.full(300, t_step, dtype=int))
    ax.hist(xt.ravel(), bins=40, density=True, color='#4d96ff', alpha=0.8)
    ax.set_title(f't={t_step}', color='white', fontsize=10)
    ax.set_xlim(-5, 5); ax.set_ylim(0, 1.5)

# Training loss
axes[1][0].plot(np.convolve(losses, np.ones(30)/30, mode='valid'),
                color='#ff6b6b', lw=2)
axes[1][0].set_title('Training Loss', color='white', fontsize=10)

# Generated samples at different training checkpoints
for ax, label, color in zip(axes[1][1:], ['Generated', 'Real target', 'Overlap'],
                              ['#6bcb77', 'white', '#ffd93d']):
    pass

samples = model.sample(500)
axes[1][1].hist(target.ravel(), bins=50, density=True, color='white',  alpha=0.6, label='Real')
axes[1][1].hist(samples,        bins=50, density=True, color='#6bcb77', alpha=0.7, label='Generated')
axes[1][1].set_title('Generated vs Real', color='white', fontsize=10)
axes[1][1].legend(fontsize=8); axes[1][1].set_xlim(-5, 5)

# Noise schedule visualization
ts = np.arange(200)
axes[1][2].fill_between(ts, 0, np.sqrt(1 - model.alpha_bar), color='#ff6b6b', alpha=0.8, label='Noise level')
axes[1][2].fill_between(ts, 0, np.sqrt(model.alpha_bar), color='#4d96ff', alpha=0.8, label='Signal level')
axes[1][2].set_title('Noise Schedule', color='white', fontsize=10)
axes[1][2].set_xlabel('Timestep t'); axes[1][2].legend(fontsize=8)

axes[1][3].text(0.5, 0.5,
    "Diffusion\nKey Insight:\n\nForward: add noise\n(fixed)\n\nReverse: denoise\n(learned network)",
    ha='center', va='center', color='white', fontsize=10,
    transform=axes[1][3].transAxes)
axes[1][3].axis('off')

plt.tight_layout(); plt.show()
print(f"Target:    N(-2,0.4) + N(2,0.4)")
print(f"Generated: mean={samples.mean():.3f}, std={samples.std():.3f}")


---
## 10. 🎮 Reinforcement Learning Networks (DQN / Policy Gradient)

### What is it?
RL networks learn by interacting with an **environment** and receiving **rewards**.  
No labeled data — the agent discovers optimal behavior through trial and error.

### Core Concepts
| Term | Meaning |
|---|---|
| **Agent** | The learning entity (the neural network) |
| **Environment** | The world the agent acts in |
| **State (s)** | Current observation of the environment |
| **Action (a)** | What the agent does |
| **Reward (r)** | Signal telling how good the action was |
| **Policy π(a|s)** | Probability distribution over actions given state |
| **Value V(s)** | Expected cumulative reward from state s |
| **Q-value Q(s,a)** | Expected cumulative reward from state s, action a |

### DQN (Deep Q-Network)
Learn `Q(s,a)` with a neural network.  
Select action: `a = argmax_a Q(s,a)`  
Train with Bellman equation:  
`Q(s,a) = r + γ · max_{a'} Q(s', a')`

Key tricks: Experience Replay, Target Network, ε-greedy exploration

### REINFORCE (Policy Gradient)
Directly learn the policy `π_θ(a|s)`.  
Loss: `L = -E[log π_θ(a|s) · G_t]`  
where `G_t` = discounted return from step t.

### When to use RL
- Game playing (Chess, Go, Atari, StarCraft)  
- Robotics and control  
- RLHF for LLM alignment  
- Trading, resource management, recommendation


In [ ]:
# ── DQN on a custom Grid World environment ────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import deque

# ── Grid World ─────────────────────────────────────────────────────────────
class GridWorld:
    """
    5×5 grid. Agent starts at (0,0), goal at (4,4), one wall.
    Actions: 0=UP, 1=DOWN, 2=LEFT, 3=RIGHT
    Reward: +10 goal, -1 wall/boundary hit, -0.1 per step
    """
    def __init__(self, size=5):
        self.size = size
        self.walls = {(1,1),(1,2),(1,3),(2,3),(3,3)}
        self.goal  = (size-1, size-1)
        self.reset()

    def reset(self):
        self.pos = (0, 0)
        return self._state()

    def _state(self):
        s = np.zeros(self.size * self.size)
        s[self.pos[0] * self.size + self.pos[1]] = 1.0
        return s

    def step(self, action):
        r, c = self.pos
        moves = [(-1,0),(1,0),(0,-1),(0,1)]
        nr, nc = r + moves[action][0], c + moves[action][1]
        if (0 <= nr < self.size and 0 <= nc < self.size and (nr,nc) not in self.walls):
            self.pos = (nr, nc)
            reward = -0.1
        else:
            reward = -1.0
        done = (self.pos == self.goal)
        if done: reward = 10.0
        return self._state(), reward, done

# ── DQN (no framework) ─────────────────────────────────────────────────────
def relu(x):    return np.maximum(0, x)
def relu_d(x):  return (x > 0).astype(float)

class DQNetwork:
    def __init__(self, state_dim, n_actions, hidden=32, lr=0.01):
        k = 0.1
        self.W1 = np.random.randn(state_dim, hidden) * k
        self.b1 = np.zeros(hidden)
        self.W2 = np.random.randn(hidden, hidden) * k
        self.b2 = np.zeros(hidden)
        self.W3 = np.random.randn(hidden, n_actions) * k
        self.b3 = np.zeros(n_actions)
        self.lr = lr

    def forward(self, x):
        self.x  = np.atleast_2d(x)
        self.h1 = relu(self.x  @ self.W1 + self.b1)
        self.h2 = relu(self.h1 @ self.W2 + self.b2)
        self.q  = self.h2 @ self.W3 + self.b3
        return self.q

    def update(self, x, q_target):
        q = self.forward(x)
        loss = np.mean((q - q_target) ** 2)
        d    = 2 * (q - q_target) / x.shape[0] if x.ndim > 1 else 2*(q - q_target)
        dW3  = self.h2.T @ d
        dh2  = (d @ self.W3.T) * relu_d(self.h2)
        dW2  = self.h1.T @ dh2
        dh1  = (dh2 @ self.W2.T) * relu_d(self.h1)
        dW1  = self.x.T  @ dh1
        for w, dw in [('W1',dW1),('W2',dW2),('W3',dW3),
                       ('b1',dh1.mean(0)),('b2',dh2.mean(0)),('b3',d.mean(0) if d.ndim>1 else d.ravel())]:
            setattr(self, w, getattr(self, w) - self.lr * dw)
        return loss

# ── Training ───────────────────────────────────────────────────────────────
np.random.seed(42)
env       = GridWorld(size=5)
STATE_DIM = env.size * env.size
N_ACTIONS = 4
GAMMA     = 0.95
EPSILON   = 1.0
EPS_DECAY = 0.995
EPS_MIN   = 0.05

dqn    = DQNetwork(STATE_DIM, N_ACTIONS, hidden=32, lr=0.02)
memory = deque(maxlen=500)   # replay buffer

episode_rewards, episode_steps = [], []

for ep in range(400):
    s = env.reset()
    total_reward, steps = 0, 0
    for _ in range(80):   # max steps per episode
        # ε-greedy action selection
        if np.random.rand() < EPSILON:
            a = np.random.randint(N_ACTIONS)
        else:
            a = dqn.forward(s).argmax()

        s_next, r, done = env.step(a)
        memory.append((s, a, r, s_next, done))
        s = s_next; total_reward += r; steps += 1
        if done: break

    # Experience replay
    if len(memory) >= 32:
        batch = [memory[i] for i in np.random.choice(len(memory), 32, replace=False)]
        for s_b, a_b, r_b, sn_b, d_b in batch:
            q_vals = dqn.forward(s_b).copy()
            q_next = dqn.forward(sn_b).max()
            q_vals[0, a_b] = r_b if d_b else r_b + GAMMA * q_next
            dqn.update(s_b, q_vals)

    EPSILON = max(EPS_MIN, EPSILON * EPS_DECAY)
    episode_rewards.append(total_reward)
    episode_steps.append(steps)

# ── Visualize learned Q-values as policy arrows ────────────────────────────
size  = env.size
arrow_dirs = [(-0.3,0),(0.3,0),(0,-0.3),(0,0.3)]   # UP DOWN LEFT RIGHT
action_sym = ['↑','↓','←','→']

grid_val = np.zeros((size, size))
best_act  = np.zeros((size, size), dtype=int)
for r in range(size):
    for c in range(size):
        if (r,c) not in env.walls and (r,c) != env.goal:
            s = np.zeros(size*size); s[r*size+c] = 1.0
            qs = dqn.forward(s)[0]
            grid_val[r,c]  = qs.max()
            best_act[r,c]  = qs.argmax()
        elif (r,c) == env.goal:
            grid_val[r,c] = 10.0

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('DQN Agent — Grid World Navigation', color='white', fontsize=14)

# Episode rewards
smooth = lambda v, n=20: np.convolve(v, np.ones(n)/n, mode='valid')
axes[0].plot(episode_rewards, color='#556', alpha=0.4, lw=1)
axes[0].plot(smooth(episode_rewards), color='#f7971e', lw=2.5, label='Smoothed reward')
axes[0].axhline(0, color='white', ls='--', alpha=0.4)
axes[0].set_title('Episode Rewards', color='white')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Total Reward')
axes[0].legend()

# Q-value heatmap
im = axes[1].imshow(grid_val, cmap='YlGn', vmin=-2, vmax=10)
for r in range(size):
    for c in range(size):
        if (r,c) in env.walls:
            axes[1].add_patch(plt.Rectangle((c-0.5,r-0.5),1,1,color='#333'))
            axes[1].text(c,r,'■',ha='center',va='center',color='gray',fontsize=14)
        elif (r,c) == env.goal:
            axes[1].text(c,r,'★',ha='center',va='center',color='gold',fontsize=16)
        else:
            sym = action_sym[best_act[r,c]]
            axes[1].text(c,r,sym,ha='center',va='center',color='white',fontsize=14,fontweight='bold')
        axes[1].text(c, r+0.35, f'{grid_val[r,c]:.1f}', ha='center', va='center',
                     color='#aaa', fontsize=7)
axes[1].set_title('Learned Policy & Q-values\n(arrows=best action, ★=goal)', color='white')
axes[1].set_xticks([]); axes[1].set_yticks([])
plt.colorbar(im, ax=axes[1], shrink=0.8)

# Steps per episode
axes[2].plot(episode_steps, color='#556', alpha=0.4, lw=1)
axes[2].plot(smooth(episode_steps), color='#56ccf2', lw=2.5)
axes[2].set_title('Steps per Episode (fewer = better)', color='white')
axes[2].set_xlabel('Episode'); axes[2].set_ylabel('Steps')

plt.tight_layout(); plt.show()

final_wins = sum(1 for r in episode_rewards[-50:] if r > 5)
print(f"\nDQN Training Summary:")
print(f"  Episodes:       400")
print(f"  Final ε:        {EPSILON:.3f}")
print(f"  Wins (last 50): {final_wins}/50 episodes")
print(f"  Avg reward (last 50): {np.mean(episode_rewards[-50:]):.2f}")


---
## 📊 Summary — All Neural Network Types at a Glance

| # | Network | Key Mechanism | Training Signal | Typical Use |
|---|---|---|---|---|
| 1 | **MLP** | Weighted sums + activation | Backprop (MSE/CE) | Tabular, classification |
| 2 | **CNN** | Convolution + pooling | Backprop | Images, spatial data |
| 3 | **RNN** | Recurrent hidden state | BPTT | Short sequences |
| 4 | **LSTM** | Cell state + 4 gates | BPTT | Long sequences, NLP |
| 5 | **Transformer** | Self-attention (Q,K,V) | Backprop | LLMs, vision, all modalities |
| 6 | **Autoencoder** | Compress → reconstruct | Reconstruction loss | Denoising, anomaly detection |
| 7 | **GAN** | Generator vs Discriminator | Adversarial (minimax) | Image synthesis |
| 8 | **GNN** | Message passing on graph | Backprop | Molecules, social networks |
| 9 | **Diffusion** | Learn to denoise | MSE on noise prediction | Image/audio generation |
| 10 | **RL Network** | Policy / Q-value learning | Rewards from environment | Games, robotics, RLHF |

---
## 🔑 Choosing the Right Architecture

```
Data Type?
├── Tabular/structured  ──────────────────→ MLP
├── Image/2D spatial    ──────────────────→ CNN (or ViT for large-scale)
├── Sequence (short)    ──────────────────→ RNN / GRU
├── Sequence (long/NLP) ──────────────────→ LSTM or Transformer
├── General language    ──────────────────→ Transformer (GPT/BERT style)
├── Graph structure     ──────────────────→ GNN (GCN/GAT/GIN)
└── Generation task?
    ├── Continuous (images, audio) ──────→ Diffusion or GAN
    ├── Discrete (text)    ──────────────→ Transformer (decoder-only)
    └── Compress/detect anomaly  ────────→ Autoencoder / VAE

Learning paradigm?
├── Labeled data available ──────────────→ Supervised (MLP, CNN, etc.)
├── No labels, want features ────────────→ Autoencoder / Self-supervised
├── Trial-and-error / reward signal ─────→ RL Network (DQN, PPO)
└── Generate new data  ──────────────────→ GAN / Diffusion / VAE
```
